In [1]:
import sys
import os
from pathlib import Path
import yaml

In [2]:
# # 1. Ensure Python understands the project root so it can import modules from src
# project_root = Path(os.getcwd())
# # If running inside the notebooks folder, move up one level
# if project_root.name == "notebooks":
#     project_root = project_root.parent
# if str(project_root) not in sys.path:
#     sys.path.append(str(project_root))


In [3]:
from src.core.parser import HiveScriptParser
from src.transformers.basic_pyspark_transformer import BasicPySparkTransformer
from src.jinja.environment import render_template
from src.paths import *

In [4]:
# script_name = "com_t_mhbos_m_client"
# datalake_type_subfolder = 'dml'
# datalake_layer_subfolder = script_name.split("_")[0]
#
# # sql_file_path = PROJECT_ROOT / "samples" / "input" / "ddl" / "raw" / f"{script_name}.sql"
# sql_file_path = DATALAKE_SCRIPT_DIR / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.sql"
# output_file_path = PROJECT_ROOT / "samples" / "converted" / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.py"
# variable_path = PROJECT_ROOT / "configs" / "rules" / "variable.yaml"

In [5]:
from utils.file_utils import parse_file_name

parse_file_name("exc_missing_trx_toms_count")

('exc', None, 'missing', 'trx')

In [6]:
def transform_and_export(script_path, transformer_class):
    print("=== STARTING PIPELINE TEST RUN ===")
    #
    # script_path = Path(script_path)
    # script_name = script_path.name
    # script_parent = script_path.parent.name
    # script_grandparent = script_path.parent.parent.name

    # Path to the SQL file to parse
    sql_file_path = script_path

    # 3. Parse the SQL file into Context (Single Source of Truth)
    print(f"[2] Reading and parsing file: {sql_file_path.name}...")
    context = HiveScriptParser.parse_file(str(sql_file_path))
    print(f"    - Extracted Source: {context.source_name}, Table: {context.table_name}")
    print(f"    - Detected {len(context.ast_nodes)} SQL statements (AST nodes)")

    # 4. Initialize the Transformer and convert the AST structure
    print("[3] Starting AST transformation (variable wrapping, dialect conversion)...")
    transformer = transformer_class()
    render_model = transformer.transform(context)
    print(f"    - Is the table partitioned? -> {render_model.is_partitioned}")

    # 5. Render the data into the Jinja Template
    print("[4] Rendering data into the Jinja Template...")
    python_script = render_template(
        template_name="pyspark/pyspark_basic.jinja",
        render_model=render_model
    )

    # 2. Load mapping configuration from YAML
    output_file_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(python_script)


    print("[5] Rendering DAGs (optimized_pyspark.jinja)...")
    dag = render_template(
        # template_name="pyspark/optimized_pyspark.jinja",
        template_name="migration/dags/dag_cur.jinja",
        render_model=render_model
    )

    # 2. Load mapping configuration from YAML
    dag_output = PROJECT_ROOT / "output" / "migration" / "dags" / "cur" / f"dag_{script_path.stem}.py"
    dag_output.parent.mkdir(parents=True, exist_ok=True)
    with open(dag_output, 'w', encoding='utf-8') as f:
        f.write(dag)


    print("\n" + "=" * 50)
    print(f"RESULT: COMPLETE PYTHON FILE SAVED AT {output_file_path}")
    print("=" * 50 + "\n")


In [10]:

table_list = [
    # Từ ảnh 1
    # "log_field_value_change_dim_account",
    # "log_field_value_change_dim_account_bank",
    # "log_field_value_change_dim_account_cif",
    # "log_field_value_change_dim_address",
    # "log_field_value_change_dim_contact_email",
    # "log_field_value_change_dim_contact_mobile",
    # "log_field_value_change_dim_contact_office",
    # "fact_sbl_loan_position",
    # "fact_margin_position",
    # "fact_loan_payment_schedule_lms_loantranchechild2",
    # "exc_missing_trx_toms_count",
    # "exc_missing_trx_smf_count",
    # "exc_missing_trx_rak_count",
    # "exc_missing_trx_mhbos_t_rec_count",
    # "exc_missing_trx_mhbos_t_payt_count",
    # "exc_missing_trx_mhbos_t_ledger_count",
    # "exc_missing_trx_mhbos_t_glled_count",
    # "exc_missing_trx_mhbos_t_ctr_count",
    # "exc_missing_trx_mhbos_t_contract_count",
    # "exc_missing_trx_m21_o_count",
    # "exc_missing_trx_m21_a_count",
    # "exc_missing_trx_lms_count",
    # "exc_missing_acc_toms",
    # "exc_missing_acc_sbl",
    # "exc_missing_acc_rak",
    # "exc_missing_acc_mhbos",
    # "exc_missing_acc_m21_o",
    # "exc_missing_acc_m21_a",
    # "exc_missing_acc_lms",


    # "exc_missing_trx_m21_o",
    # "exc_missing_acc_m21_a",
    # "exc_missing_acc_lms",
    # 
    # # Từ ảnh 2
    # "exc_missing_acc_kdi",
    # "exc_customer_id_mapping"
    #
    #  "cur_dim_indicators_rak",
    # "cur_dim_indicators_kdi",
    # "cur_dim_indicators_mhbos",
    # "cur_dim_indicators_lms",
    # "cur_dim_indicators_m21",
    # "cur_dim_indicators_toms_eretail",
    # "cur_dim_indicators_toms_ecorporate",
    # "cur_dim_indicators_sbl",


    "cur_einvoice_account"

]

for table in table_list:
    output_file_path = PROJECT_ROOT / "output" / "migration" / "dml" / "cur" / "simple" /  f"{table}.py"
    transform_and_export(DATALAKE_SCRIPT_DIR / "dml" / "cur" / f"{table}.sql", BasicPySparkTransformer)



=== STARTING PIPELINE TEST RUN ===
[2] Reading and parsing file: cur_einvoice_account.sql...
    - Extracted Source: einvoice, Table: account
    - Detected 2 SQL statements (AST nodes)
[3] Starting AST transformation (variable wrapping, dialect conversion)...
    - Is the table partitioned? -> False
[4] Rendering data into the Jinja Template...
[5] Rendering DAGs (optimized_pyspark.jinja)...

RESULT: COMPLETE PYTHON FILE SAVED AT C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\dml\cur\simple\cur_einvoice_account.py



In [8]:
def main():


    # Table name
    transform_and_export(sql_file_path, BasicPySparkTransformer)


if __name__ == "__main__":
    main()


NameError: name 'sql_file_path' is not defined